## Imports

In [26]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    RegexTokenizer,
    StopWordsRemover,
    CountVectorizer,
    IDF,
    StringIndexer,
    ChiSqSelector)

## Global Variables

In [89]:
INPUT_PATH = "hdfs:///dic_shared/amazon-reviews/full/reviews_devset.json"
OUTPUT_PATH = f"../result/output_tfidf.txt"

NUM_TOP_TERMS = 2000

## Environment Setup

In [28]:
spark = (SparkSession.builder
    .master("local[*]")
    .appName("Part2_TFIDF")
    .getOrCreate()
        )

In [29]:
reviews = spark.read.json(INPUT_PATH)

## Dataset Preparation

In [69]:
df = reviews.select(
    col("category"),
    col("reviewText")
).na.drop(subset=["category", "reviewText"])

In [70]:
df.head(5)

[Row(category='Patio_Lawn_and_Garde', reviewText="This was a gift for my other husband.  He's making us things from it all the time and we love the food.  Directions are simple, easy to read and interpret, and fun to make.  We all love different kinds of cuisine and Raichlen provides recipes from everywhere along the barbecue trail as he calls it. Get it and just open a page.  Have at it.  You'll love the food and it has provided us with an insight into the culture that produced it. It's all about broadening horizons.  Yum!!"),
 Row(category='Patio_Lawn_and_Garde', reviewText='This is a very nice spreader.  It feels very solid and the pneumatic tires give it great maneuverability and handling over bumps.  The control arm is solid metal, not a cable, which gives you precise control and will last a long time.  The settings take some experimentation with your various products to get it right, but that is true of any spreader.  It has good distribution... probably flings material a little 

## Tokenization

In [71]:
token_split_re = r"[\s\d\(\)\[\]\{\}\.!?;,:+\-_\"'`~#@&\*%€\$\\/]+"

tokenizer = RegexTokenizer(
    inputCol="reviewText",
    outputCol="tokens",
    pattern=token_split_re,
    gaps=True,
    toLowercase=True,
    minTokenLength=1
)

## Stop Words

In [73]:
remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

## Vectorizor

In [76]:
count_vectorizer = CountVectorizer(
    inputCol="filtered_tokens",
    outputCol="tf_features",
    minDF=1.0
)

## TF-IDF

In [77]:
idf = IDF(
    inputCol="tf_features",
    outputCol="tfidf_features"
)

## Indexer

In [78]:
label_indexer = StringIndexer(
    inputCol="category",
    outputCol="label"
)

## Chi Selector

In [79]:
chi_sq_selector = ChiSqSelector(
    numTopFeatures=NUM_TOP_TERMS,
    featuresCol="tfidf_features",
    outputCol="selected_features",
    labelCol="label"
)

## Pipeline

In [80]:
pipeline = Pipeline(stages=[
    tokenizer,
    remover,
    count_vectorizer,
    idf,
    label_indexer,
    chi_sq_selector
])

In [81]:
pipeline_model = pipeline.fit(df)

26/05/16 22:32:41 WARN DAGScheduler: Broadcasting large task binary with size 1069.1 KiB
26/05/16 22:32:50 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/05/16 22:32:50 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/05/16 22:32:57 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
                                                                                

In [82]:
transformed = pipeline_model.transform(df)

transformed.limit(5).show()

26/05/16 22:34:54 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB


+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----+--------------------+
|            category|          reviewText|              tokens|     filtered_tokens|         tf_features|      tfidf_features|label|   selected_features|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----+--------------------+
|Patio_Lawn_and_Garde|This was a gift f...|[this, was, a, gi...|[gift, husband, m...|(96675,[5,7,8,9,2...|(96675,[5,7,8,9,2...| 18.0|(2000,[5,7,8,9,27...|
|Patio_Lawn_and_Garde|This is a very ni...|[this, is, a, ver...|[nice, spreader, ...|(96675,[2,4,8,9,1...|(96675,[2,4,8,9,1...| 18.0|(2000,[2,4,8,9,16...|
|Patio_Lawn_and_Garde|The metal base wi...|[the, metal, base...|[metal, base, hos...|(96675,[6,20,21,3...|(96675,[6,20,21,3...| 18.0|(2000,[6,19,20,37...|
|Patio_Lawn_and_Garde|For the most part...|[for, the, most, ...|[part,

In [84]:
pipeline_model.stages

[RegexTokenizer_9e06f033fcf8,
 StopWordsRemover_1e9f1331ee35,
 CountVectorizerModel: uid=CountVectorizer_e33c80b22745, vocabularySize=96675,
 IDFModel: uid=IDF_ed2c4ad41e6b, numDocs=78829, numFeatures=96675,
 StringIndexerModel: uid=StringIndexer_de6e61a68c1d, handleInvalid=error,
 ChiSqSelectorModel: uid=ChiSqSelector_efe86d1e5953, numSelectedFeatures=2000]

In [85]:
count_vectorizer_model = pipeline_model.stages[2]
chi_sq_selector_model = pipeline_model.stages[5]

vocabulary = count_vectorizer_model.vocabulary
selected_indices = chi_sq_selector_model.selectedFeatures

selected_terms = [vocabulary[index] for index in selected_indices]

In [86]:
selected_terms = sorted(selected_terms)

## Output

In [90]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for term in selected_terms:
        f.write(term + "\n")

print("Number of selected terms:", len(selected_terms))
print("Selected terms written to:", OUTPUT_PATH)

Number of selected terms: 2000
Selected terms written to: ../result/output_tfidf.txt


In [88]:
spark.stop()